# 在自定义数据集上微调GIT模型进行图像描述

在本笔记本中，我们将在一个小型图像描述数据集上微调[GIT](https://huggingface.co/docs/transformers/main/en/model_doc/git)（GenerativeImage2Text的缩写）模型。

GIT在编写本文时是最先进的图像/视频描述和问答（QA）模型。

## 环境设置
**环境配置：**

1. MindSpore 2.3.0
2. Mindnlp 0.3.1
3. Python 3.9



**使用华为云 ModelArts 作为AI平台**

在环境搭建部分，使用了AI gallery社区中相关mindnlp项目搭建mindnlp环境的代码。

### 环境配置

配置python3.9环境

In [ ]:
%%capture captured_output
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

*注：以上代码执行完成后，需点击左上角或右上角将kernel更换为python-3.10.0*

2. 安装mindspore2.3.0，安装指南详见：[MindSpore安装](https://www.mindspore.cn/install/)

3. 安装MindNLP及相关依赖，MindNLP官方仓详见：[MindNLP](https://github.com/mindspore-lab/mindnlp)

In [ ]:
!pip install mindspore==2.3.0
!pip install mindnlp==0.3.1
!pip install -q decord
!pip install ipywidgets

## 创建图像描述数据集

接下来，我们将创建一个小型图像描述数据集，该数据集由（图像，文本）对组成。

作为一名足球迷，我简单地访问了几位最著名的足球运动员的维基百科页面，并从各自的页面中获取了带有说明文字的图像。

我们将按照[这里](https://huggingface.co/docs/datasets/main/en/image_dataset#image-captioning)的指南创建一个🤗 Dataset，它允许非常快速的处理。基本上，我们需要在包含图像的文件夹中添加一个metadata.jsonl文件。这个元数据文件包含了每张图像的描述文本。


In [ ]:
!git clone https://github.com/wjy4399/Toy_dataset.git

In [ ]:
import json
captions = [{"file_name": "ronaldo.jpeg", "text": "Ronaldo with Portugal at the 2018 World Cup"},
{"file_name": "messi.jpeg", "text": "Messi with Argentina at the 2022 FIFA World Cup"},
{"file_name": "zidane.jpeg", "text": "Zinédine Zidane pendant la finale de la Coupe du monde 2006."},
{"file_name": "maradona.jpeg", "text": "Maradona after winning the 1986 FIFA World Cup with Argentina"},
{"file_name": "ronaldo_.jpeg", "text": "Ronaldo won La Liga in his first season and received the Pichichi Trophy in his second."},
{"file_name": "pirlo.jpeg", "text": "Pirlo with Juventus in 2014"},]
# path to the folder containing the images
root = "Toy_dataset/"
# add metadata.jsonl file to this folder
with open(root + "metadata.jsonl", 'w') as f:
    for item in captions:
        f.write(json.dumps(item) + "\n")

接下来，我们使用ImageFolder功能快速将其转换为🤗 Dataset。我们将指定这只是数据集的训练分割部分。

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imagefolder", data_dir=root, split="train")

让我们检查数据集是否创建正确：

In [ ]:
dataset

让我们看一个例子：

In [ ]:
example = dataset[0]
image = example["image"]
width, height = image.size
display(image.resize((int(0.3*width), int(0.3*height))))

让我们检查它对应的描述文本：

In [ ]:
example["text"]

## 创建MindSpore数据集

接下来，我们创建一个标准的[Mindspore数据集]。数据集的每个项目返回模型所需的输入，在这种情况下是`input_ids`、`attention_mask`和`pixel_values`。

我们使用`GitProcessor`将每个（图像，文本）对转换为所需的输入。基本上，文本被转换为`input_ids`和`attention_mask`，而图像被转换为`pixel_values`。


In [ ]:
class ImageCaptioningDataset:
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor
        
        self.encoded_inputs_list = []
        self.__get_encoded_inputs_list__()
    
    def __get_encoded_inputs_list__(self):
        for idx in range(self.__len__()):
            item = self.dataset[idx]
            
            encoding = self.processor(images=item["image"], text=item["text"], 
                                     padding="max_length", return_tensors="np")
            
            # 移除批次维度
            encoding = {k:v.squeeze() for k,v in encoding.items()}
            
            # 将字典值转换为元组，以便与MindSpore数据集兼容
            self.encoded_inputs_list.append(tuple(encoding.values()))
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # 直接返回预处理好的数据
        return self.encoded_inputs_list[idx]

In [ ]:
from mindnlp.transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("microsoft/git-base-coco")

In [ ]:
processor

In [ ]:
train_dataset = ImageCaptioningDataset(dataset, processor)

让我们检查数据集的一个例子：

In [ ]:
encoded_inputs = train_dataset[0]

In [ ]:
col_names = ['input_ids', 'attention_mask', 'pixel_values']

In [ ]:
for k,v in zip(col_names, encoded_inputs):
    print(k, v.shape, v.dtype)

## 创建MindSpore DataLoader

接下来，我们创建一个相应的MindSpore DataLoader，它允许我们从数据集中获取数据批次。

我们需要这个，因为神经网络（如GIT）是在数据批次上进行训练的，使用随机梯度下降（SGD）方法。


In [ ]:
print(processor.tokenizer.decode(encoded_inputs[0]))

In [ ]:
from mindspore.dataset import GeneratorDataset

train_dataloader = GeneratorDataset(train_dataset, shuffle=True, column_names=col_names)

In [ ]:
batch_size = 2
train_dataloader = train_dataloader.batch(batch_size)
train_dataloader.get_col_names()

In [ ]:
dict_iterator0 = train_dataloader.create_dict_iterator()
datas = next(dict_iterator0)
for k,v in datas.items():
    print(k, v.shape, v.dtype)

让我们检查一个批次，并进行一些合理性检查。例如，我们可以将input_ids解码回文本：

In [ ]:
processor.decode(datas["input_ids"][0])

我们可以“去归一化”像素值以恢复图像：

In [ ]:
from PIL import Image
import numpy as np

MEAN = np.array([123.675, 116.280, 103.530]) / 255
STD = np.array([58.395, 57.120, 57.375]) / 255

unnormalized_image = (datas["pixel_values"][0].numpy() * np.array(STD)[:, None, None]) + np.array(MEAN)[:, None, None]
unnormalized_image = (unnormalized_image * 255).astype(np.uint8)
unnormalized_image = np.moveaxis(unnormalized_image, 0, -1)
Image.fromarray(unnormalized_image)

看起来不错！检查你的数据总是很重要 ;) 有关训练神经网络时的提示，请参阅这篇[博客](http://karpathy.github.io/2019/04/25/recipe/) ，其中提供了很好的概述。

## 定义模型

接下来，我们实例化一个模型。我们从预训练的GIT-base模型开始（该模型已经由微软在400万对图像-文本上进行了预训练）。

当然，您也可以从[模型库](https://huggingface.co/models?other=git)开始微调其他GIT模型。


In [ ]:
from mindnlp.transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("microsoft/git-base-coco")

## 虚拟前向传播

检查批次上的初始损失总是很好的做法。请参阅上面的博客。


In [ ]:
outputs = model(input_ids=datas["input_ids"],
                attention_mask=datas["attention_mask"],
                pixel_values=datas["pixel_values"],
                labels=datas["input_ids"])
outputs.loss

## 训练模型

接下来，让我们训练模型！我们在这里使用原生的MindSpore。

由于我创建了一个非常小的数据集仅用于演示目的，我们将让模型过拟合这个数据集。如果它能够过拟合（即达到零损失），那么这就是确认一切正常工作的好方法。请参阅上面的博客。


In [ ]:
from mindnlp.core.optim import AdamW
optimizer = AdamW(model.trainable_params(), lr =5e-5)

global_step = 0
num_train_epochs = 50

In [ ]:
model.train()

In [ ]:
from mindnlp.core.autograd import value_and_grad

def forward_fn(batch):
    # get the inputs;
    input_ids = batch['input_ids']
    pixel_values=batch["pixel_values"]
    attention_mask = batch["attention_mask"] 
    outputs = model(input_ids=input_ids,
                    pixel_values=pixel_values,
                    attention_mask=attention_mask,
                    labels=input_ids) 
    loss = outputs.loss
    
    return loss

grad_fn = value_and_grad(forward_fn, model.trainable_params(), attach_grads=True)

In [ ]:
from tqdm import tqdm

# put the model in training mode
model.set_train(True)

for epoch in range(num_train_epochs):  
    print("Epoch:", epoch)
    for batch in tqdm(train_dataloader.create_dict_iterator()):
        optimizer.zero_grad()
        # forward, backward + optimize
        loss = grad_fn(batch)
        optimizer.step()
        # print loss every 1 steps
        if global_step % 1 == 0:
            print(f"Loss after {global_step} steps: {loss.item()}")

        global_step += 1

## 推理

现在我们已经训练了模型，让我们加载马拉多纳的图像并对其进行推理。


In [ ]:
# load image
example = dataset[3]
image = example["image"]
width, height = image.size
display(image.resize((int(0.9*width), int(0.9*height))))

In [ ]:
inputs = processor(images=image, return_tensors="ms")
pixel_values = inputs.pixel_values
generated_ids = model.generate(pixel_values=pixel_values, max_length=80)
generated_caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_caption)

太好了！我们已经成功地在我们的小型（图像，文本）数据集上微调了GIT，以生成足球运动员图像的描述。